In [1]:
import numpy as np
import pandas as pd
import zarr
import torch
import math
from datetime import date, datetime, timedelta
import statsmodels.api as sm
import os
import gc
import re
import shutil
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
save_path = "figure2/"

os.makedirs(save_path, exist_ok=True)

ds = zarr.open_group("/data_2/scratch/sbiegel/processed/ndvi_dataset_temporal.zarr",mode = "r")
params = ds["params"]
params_lower = params["params_lower"]
params_upper = params["params_upper"]
ndvi = ds["ndvi"]
ndsi = ds["ndsi"]
dates = pd.to_datetime([d.decode("utf-8") for d in ds["dates"][:]])

T_SCALE = 1.0 / 365.0
doy = dates.dayofyear
t = torch.tensor(doy * T_SCALE, dtype=torch.float32)



order = np.argsort(dates)
dates_sorted = np.array(dates)[order]
t_sorted = t[order]

# Raster info
height, width = 24542, 37728
left, bottom = 2474090.0, 1065110.0
px = 10.0
top = bottom + height * px

mask_path = "/data_2/scratch/sbiegel/processed/forest_mask.npy"


def extract_pixel(UL_x, UL_y,BR_x, BR_y ):

    # ----- compute pixel window (row 0 = top) -----
    x_min, x_max = min(UL_x, BR_x), max(UL_x, BR_x)
    y_min, y_max = min(UL_y, BR_y), max(UL_y, BR_y)

    col_min = int(math.floor((x_min - left) / px))
    col_max = int(math.floor((x_max - left) / px))

    row_min = int(math.floor((top - y_max) / px))
    row_max = int(math.floor((top - y_min) / px))

    # clip to bounds
    col_min = max(0, min(width - 1, col_min))
    col_max = max(0, min(width - 1, col_max))
    row_min = max(0, min(height - 1, row_min))
    row_max = max(0, min(height - 1, row_max))

    win_cols = col_max - col_min + 1
    win_rows = row_max - row_min + 1
    print(f"Window cols {col_min}..{col_max} ({win_cols}), rows {row_min}..{row_max} ({win_rows})")

    # ----- load mask -----
    mask = np.load(mask_path)
    assert mask.shape == (height, width), f"Mask shape {mask.shape} != raster {(height, width)}"

    mask_flat = mask.ravel(order="C")
    masked_positions = np.flatnonzero(mask_flat)
    n_masked = masked_positions.size
    print(f"Mask has {n_masked} True pixels.")

    # build index map from full array -> masked array
    idx_map = np.full(mask_flat.shape[0], -1, dtype=np.int64)
    idx_map[masked_positions] = np.arange(n_masked, dtype=np.int64)

    # ----- compute flat indices in window -----
    rows = np.arange(row_min, row_max + 1, dtype=np.int64)
    cols = np.arange(col_min, col_max + 1, dtype=np.int64)
    rr, cc = np.meshgrid(rows, cols, indexing="ij")
    full_flat_idx = (rr * width + cc).ravel()

    masked_idx_in_window = idx_map[full_flat_idx]
    is_masked = masked_idx_in_window >= 0
    n_masked_in_window = is_masked.sum()
    print(f"Pixels in window: {full_flat_idx.size}, masked pixels: {n_masked_in_window}")

    if n_masked_in_window == 0:
        raise RuntimeError("No masked pixels in window!")

    sel = masked_idx_in_window[is_masked].tolist()


    return(sel)


# save all the performance metrics
all_metrics = []

# storm
center_x, center_y =  2644218.94, 1134325.81

UL_x, UL_y = center_x - 200, center_y - 200 
BR_x, BR_y = center_x + 200, center_y + 200
sel_1 = extract_pixel( UL_x = UL_x, UL_y = UL_y, BR_x = BR_x, BR_y = BR_y) 

/home/francesco/miniconda3/envs/ndvi/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:99: UserWarning: The codec `vlen-bytes` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


Window cols 16992..17032 (41), rows 17600..17640 (41)
Mask has 105715396 True pixels.
Pixels in window: 1681, masked pixels: 1628


In [3]:
dates[360:380]

DatetimeIndex(['2021-08-19', '2021-08-27', '2021-06-21', '2021-08-15',
               '2021-08-12', '2021-07-10', '2021-07-22', '2021-06-01',
               '2021-06-23', '2021-07-02', '2021-06-11', '2021-08-17',
               '2021-07-20', '2021-07-12', '2021-06-16', '2021-07-11',
               '2021-08-26', '2021-06-15', '2021-06-17', '2021-06-18'],
              dtype='datetime64[ns]', freq=None)

In [13]:
obs_ndvi = ndvi[sel_1,364] / 10000

In [26]:
T_SCALE = 1.0 / 365.0
doy =  dates[364].dayofyear
t = torch.tensor(doy * T_SCALE, dtype=torch.float32)  
t = torch.as_tensor(t, dtype=torch.float32).flatten()

In [31]:
def double_logistic_function(t, params):
    sos, mat_minus_sos, sen, eos_minus_sen, M, m = torch.split(torch.as_tensor(params, dtype=torch.float32), 1, dim=1)
    mat_minus_sos = torch.nn.functional.softplus(mat_minus_sos)
    eos_minus_sen = torch.nn.functional.softplus(eos_minus_sen)
    sigmoid_sos_mat = torch.sigmoid(-2 * (2 * sos + mat_minus_sos - 2 * t[:, None]) / (mat_minus_sos + 1e-10))
    sigmoid_sen_eos = torch.sigmoid(-2 * (2 * sen + eos_minus_sen - 2 * t[:, None]) / (eos_minus_sen + 1e-10))
    return (M - m) * (sigmoid_sos_mat - sigmoid_sen_eos) + m


lower = double_logistic_function(t, params_lower[[4]]).squeeze().numpy()
upper = double_logistic_function(t, params_upper[[4]]).squeeze().numpy()



array(0.78254294, dtype=float32)

tensor(0.6137)